In [1]:
tabla='ctsci10'
col_essi='fec_sol'
col_tabla='solcitafec'
essi='essi_dat_cex003_etl'

In [2]:
import pandas as pd
from decouple import config
from sqlalchemy import create_engine
from datetime import datetime
from sqlalchemy import text
import oracledb
import decouple
import sys

In [3]:
# CONEXIONES
config = decouple.AutoConfig(' ')
DB_USER = config("USER2_BDI_POSTGRES")
DB_PASSWORD = config("PASS2_BDI_POSTGRES")
DB_HOST = config("HOST2_BDI_POSTGRES")
DB_PORT = "5432"
DB_NAME = "essi_etl"

cadena1 = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_NAME = "dw_essalud"
cadena2 = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_NAME = "dl_essi"
cadena3 = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [4]:
fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=3", con=connection2)
fecha= fecha.iloc[0, 0]
print(fecha)

now = datetime.now()

query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=3"

c1= text(query)
connection2.execute(c1)

2023-07-01


In [5]:
fecha='2023-06-01'

In [6]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

base1 = pd.read_sql_query(f"SELECT * FROM {tabla} WHERE {col_tabla} >= TO_DATE('{fecha}', 'YYYY-MM-DD')", con=connection0)


connection0.close()
engine0.dispose()

In [7]:
base1 = base1.rename(columns={
    'estatensolcitacod': 'cod_est',
    'solcitaactcod': 'cod_act',
    'solcitaactespcod': 'cod_sub',
    'solcitaactmedcitnum': 'act_med',
    'solcitaactmedorinum': 'act_med_ori',
    'solcitaarehoscod': 'cod_are',
    'solcitacenasicitcod': 'cas_sol',
    'solcitacenasicod': 'cod_cas',
    'solcitacenasioricod': 'cas_sol_ori',
    'solcitacpscod': 'cod_cps',
    'solcitacrefec': 'fec_cre',
    'solcitaestregcod': 'est_reg',
    'solcitafec': 'fec_sol',
    'solcitafecanu': 'fec_anu',
    'solcitafecpref': 'fec_pre',
    'solcitaipanucod': 'ip_anu',
    'solcitaipcrecod': 'ip_cre',
    'solcitaipmodcod': 'ip_mod',
    'solcitamodanucod': 'mod_anu',
    'solcitamodcrecod': 'mod_cre',
    'solcitamodfec': 'fec_mod',
    'solcitamodmodcod': 'mod_mod',
    'solcitanum': 'num_sol',
    'solcitaoricenasicitcod': 'ori_sol',
    'solcitaoricenasicod': 'ori_cas',
    'solcitaoricenasioricod': 'ori_sol_ori',
    'solcitapacsecnum': 'pac_sec',
    'solcitapricod': 'cod_pri',
    'solcitasecnum': 'sec_num',
    'solcitaservhoscod': 'cod_ser',
    'solcitatipocitacod': 'cod_tci',
    'solcitatiptercod': 'tip_ter',
    'solcitatramedfissolnum': 'med_fis',
    'solcitausuanucod': 'usu_anu',
    'solcitausucrecod': 'usu_cre',
    'solcitausumodcod': 'usu_mod',
    'turprefsolcitacod': 'cod_tpc',
    'diaprefsolcitcod': 'cod_dpc'
})

In [8]:
base2=pd.read_sql_query(f"SELECT * FROM {essi} LIMIT 10", con=connection1)

In [9]:
# Cargar los DataFrames en las tablas temporales usando pd.to_sql
cas = pd.read_sql_query(f"SELECT id_red,cod_cas,des_cas FROM dim_cas where id_red is not null", con=connection2)
red = pd.read_sql_query(f"SELECT id_red,cod_red,des_red FROM dim_red", con=connection2)
cas_red=pd.merge(cas,red,how='left',on='id_red')
base1 = pd.merge(base1, cas_red, how='inner', on='cod_cas')

In [10]:
base1.shape

(372876, 43)

In [11]:
servicios = pd.read_sql_query(f"SELECT cod_ser,des_ser FROM dim_servicios", con=connection2)
base1 = pd.merge(base1, servicios, how='inner', on='cod_ser')

In [12]:
base1.shape

(372876, 44)

In [13]:
areas = pd.read_sql_query(f"SELECT cod_are,des_are FROM dim_areas", con=connection2)
base1 = pd.merge(base1, areas, how='inner', on='cod_are')

In [14]:
base1.shape

(372876, 45)

In [15]:
subacti = pd.read_sql_query(f"SELECT des_sub,cod_sub,cod_act FROM dim_subacti", con=connection2)
subacti["KEY"]=subacti["cod_sub"]+subacti["cod_act"]
subacti=subacti.drop(["cod_sub",'cod_act'], axis=1)
base1["KEY"]=base1["cod_sub"].astype(str)+base1['cod_act'].astype(str)
base1["KEY"]=base1["KEY"].str.replace(' ', '', regex=True)
subacti["KEY"]=subacti["KEY"].str.replace(' ', '', regex=True)
base1 = pd.merge(base1,subacti,how='inner',on='KEY')
base1=base1.drop('KEY', axis=1)

In [16]:
base1.shape

(372876, 46)

In [17]:
activi = pd.read_sql_query(f"SELECT cod_act,des_act FROM dim_activi", con=connection2)
base1 = pd.merge(base1, activi, how='inner', on='cod_act')

In [18]:
base1.shape

(372876, 47)

In [19]:
tipcit = pd.read_sql_query(f"SELECT cod_tci,des_tci FROM dim_tipcit", con=connection2)
base1 = pd.merge(base1, tipcit, how='inner', on='cod_tci')

In [20]:
base1.shape

(372876, 48)

In [21]:
base1.columns

Index(['num_sol', 'fec_sol', 'ori_cas', 'cod_cas', 'pac_sec', 'cod_are',
       'cod_ser', 'cod_act', 'cod_sub', 'cod_tci', 'fec_pre', 'cod_dpc',
       'cod_tpc', 'cod_est', 'ori_sol_ori', 'cas_sol_ori', 'act_med_ori',
       'ori_sol', 'cas_sol', 'act_med', 'est_reg', 'usu_cre', 'fec_cre',
       'usu_mod', 'fec_mod', 'cod_pri', 'ip_cre', 'usu_anu', 'ip_anu',
       'fec_anu', 'ip_mod', 'mod_cre', 'mod_anu', 'mod_mod', 'sec_num',
       'cod_cps', 'solcitaindica', 'med_fis', 'tip_ter', 'id_red', 'des_cas',
       'cod_red', 'des_red', 'des_ser', 'des_are', 'des_sub', 'des_act',
       'des_tci'],
      dtype='object')

In [22]:
cexdiapresol = pd.read_sql_query(f"SELECT cod_dps,des_dps FROM dim_cexdiapresol", con=connection2)
cexdiapresol = cexdiapresol.rename(columns={'cod_dps':'cod_dpc'})
cexdiapresol = cexdiapresol.rename(columns={'des_dps':'des_dpc'})
base1 = pd.merge(base1, cexdiapresol, how='inner', on='cod_dpc')

In [23]:
base1.shape

(372876, 49)

In [24]:
cexestreg = pd.read_sql_query(f"SELECT cod_reg,des_reg FROM dim_cexestreg", con=connection2)
cexestreg = cexestreg.rename(columns={'cod_reg':'est_reg'})
base1 = pd.merge(base1, cexestreg, how='inner', on='est_reg')

In [25]:
base1.shape

(372876, 50)

In [26]:
cexestsolcit = pd.read_sql_query(f"SELECT cod_esc,des_esc FROM dim_cexestsolcit", con=connection2)
cexestsolcit = cexestsolcit.rename(columns={'cod_esc':'cod_est'})
cexestsolcit = cexestsolcit.rename(columns={'des_esc':'des_est'})
base1 = pd.merge(base1, cexestsolcit, how='inner', on='cod_est')

In [27]:
base1.shape

(372876, 51)

In [28]:
cexprisol = pd.read_sql_query(f"SELECT cod_pri,des_pr1 FROM dim_cexprisol", con=connection2)
cexprisol = cexprisol.rename(columns={'des_pr1':'des_pri'})
cexprisol['cod_pri']= cexprisol['cod_pri'].astype(int)
base1 = pd.merge(base1, cexprisol, how='inner', on='cod_pri')

In [29]:
base1.shape

(372876, 52)

In [30]:
cexturprecit = pd.read_sql_query(f"SELECT cod_tpc,des_tpc FROM dim_cexturprecit", con=connection2)
base1 = pd.merge(base1, cexturprecit, how='inner', on='cod_tpc')

In [31]:
base1.shape

(372876, 53)

In [32]:
cas = cas.rename(columns={'cod_cas':'cas_sol'})
cas = cas.rename(columns={'des_cas':'des_cso'})
cas = cas.dropna()
base1 = pd.merge(base1, cas, how='left', on='cas_sol')

In [33]:
base1.shape

(372876, 55)

In [34]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df2_columns - df1_columns
different_columns

set()

In [35]:
connection1.close()
connection2.close()
connection3.close()

In [36]:
connection1.close()
connection2.close()
connection3.close()

In [37]:
#borrando=f"DELETE FROM {essi} WHERE {col_essi} >='{fecha}'"
#borrado = connection1.execute(borrando)

In [38]:
#subir datos